# Historical experiment: testoneTransformer.ipynb
Archived source with outputs/execution counts cleared. Superseded by the numbered notebooks.
Original paths and experimental branches are retained for reference; this is not the supported run sequence.

In [ ]:
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_ID = "roneneldan/TinyStories-3M"

# Pin the exact model revision for reproducible experiments.
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

In [ ]:
model.eval()

In [ ]:
model.transformer.h[0]

In [ ]:
import numpy as np
from pathlib import Path
from torch import nn

try:
    import litert_torch
    print("LiteRT Torch version:", litert_torch.__version__)
except ImportError:
    print("LiteRT Torch is not installed.")

In [ ]:
SEQUENCE_LENGTH = 32

prompt = (
    "Once upon a time, a curious little girl named Lily lived in a "
    "quiet village beside a green forest. Every morning, she carried "
    "fresh bread to her grandmother and watched the birds in the trees."
)

encoded = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
)

available_tokens = encoded["input_ids"].shape[1]
print("Available prompt tokens:", available_tokens)

assert available_tokens >= SEQUENCE_LENGTH

input_ids = encoded["input_ids"][:, :SEQUENCE_LENGTH]

position_ids = torch.arange(
    SEQUENCE_LENGTH,
    dtype=torch.long,
).unsqueeze(0)

with torch.no_grad():
    token_embeddings = model.transformer.wte(input_ids)
    position_embeddings = model.transformer.wpe(position_ids)

    block_input = model.transformer.drop(
        token_embeddings + position_embeddings
    )

print("Input IDs shape:", input_ids.shape)
print("Block input shape:", block_input.shape)
print("Block input dtype:", block_input.dtype)
print(
    "Block input range:",
    block_input.min().item(),
    block_input.max().item(),
)

In [ ]:
class Block0Wrapper(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = block

    def forward(self, hidden_states):
        outputs = self.block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )

        return outputs[0]


block0 = Block0Wrapper(model.transformer.h[0]).eval()

with torch.no_grad():
    pytorch_output = block0(block_input)

print("Output shape:", pytorch_output.shape)
print("Output dtype:", pytorch_output.dtype)
print(
    "Output range:",
    pytorch_output.min().item(),
    pytorch_output.max().item(),
)

In [ ]:
sample_inputs = (block_input,)

exported_program = torch.export.export(
    block0,
    sample_inputs,
)

print("torch.export succeeded")
print(
    "Exported graph nodes:",
    len(list(exported_program.graph.nodes)),
)

In [ ]:
edge_model = litert_torch.convert(
    block0,
    sample_inputs,
)

print("LiteRT conversion succeeded")

In [ ]:
litert_output = edge_model(*sample_inputs)

if isinstance(litert_output, (tuple, list)):
    litert_output = litert_output[0]

litert_output = np.asarray(litert_output)
pytorch_output_np = pytorch_output.detach().cpu().numpy()

absolute_error = np.abs(
    pytorch_output_np - litert_output
)

print("LiteRT output shape:", litert_output.shape)
print("Maximum absolute error:", absolute_error.max())
print("Mean absolute error:", absolute_error.mean())

outputs_close = np.allclose(
    pytorch_output_np,
    litert_output,
    atol=1e-4,
    rtol=1e-4,
)

print("Outputs sufficiently close:", outputs_close)

In [ ]:
litert_output = edge_model(*sample_inputs)

if isinstance(litert_output, (tuple, list)):
    litert_output = litert_output[0]

litert_output = np.asarray(litert_output)
pytorch_output_np = pytorch_output.detach().cpu().numpy()

absolute_error = np.abs(
    pytorch_output_np - litert_output
)

print("LiteRT output shape:", litert_output.shape)
print("Maximum absolute error:", absolute_error.max())
print("Mean absolute error:", absolute_error.mean())

outputs_close = np.allclose(
    pytorch_output_np,
    litert_output,
    atol=1e-4,
    rtol=1e-4,
)

print("Outputs sufficiently close:", outputs_close)

In [ ]:
output_directory = Path("models")
output_directory.mkdir(exist_ok=True)

output_path = (
    output_directory
    / "tinystories_block0_fp32.tflite"
)

edge_model.export(str(output_path))

print("Saved model:", output_path.resolve())
print(
    "File size:",
    output_path.stat().st_size,
    "bytes",
)

In [ ]:
from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)

from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)

from litert_torch.quantize.quant_config import QuantConfig

In [ ]:
quantization_config = get_symmetric_quantization_config(
    is_per_channel=False,
    is_dynamic=False,
    is_qat=False,
)

quantizer = PT2EQuantizer().set_global(
    quantization_config
)

print("Static W8A8 quantizer created")
print("Dynamic quantization:", quantization_config.is_dynamic)

In [ ]:
quantized_exported_program = torch.export.export(
    block0,
    sample_inputs,
)

prepared_model = prepare_pt2e(
    quantized_exported_program.module(),
    quantizer,
)

print("PT2E preparation succeeded")

In [ ]:
with torch.no_grad():
    calibration_output = prepared_model(
        block_input
    )

print("Calibration succeeded")
print("Calibration output shape:", calibration_output.shape)

In [ ]:
quantized_model = convert_pt2e(
    prepared_model,
    fold_quantize=False,
)

print("PT2E quantization conversion succeeded")

In [ ]:
with torch.no_grad():
    quantized_torch_output = quantized_model(
        block_input
    )

print("Quantized output shape:",
      quantized_torch_output.shape)
print("Quantized output dtype:",
      quantized_torch_output.dtype)

In [ ]:
fp32_output = pytorch_output.detach().cpu()
pt2e_output = quantized_torch_output.detach().cpu()

quantization_error = fp32_output - pt2e_output

mean_absolute_error = (
    quantization_error.abs().mean().item()
)

maximum_absolute_error = (
    quantization_error.abs().max().item()
)

cosine_similarity = torch.nn.functional.cosine_similarity(
    fp32_output.flatten(),
    pt2e_output.flatten(),
    dim=0,
).item()

print("FP32 versus PT2E W8A8")
print("Mean absolute error:", mean_absolute_error)
print("Maximum absolute error:", maximum_absolute_error)
print("Cosine similarity:", cosine_similarity)

In [ ]:
quantization_nodes = []

for node in quantized_model.graph.nodes:
    target = str(node.target)

    if (
        "quantize" in target
        or "dequantize" in target
    ):
        quantization_nodes.append(
            (node.name, node.op, target)
        )

print(
    "Number of quantize/dequantize nodes:",
    len(quantization_nodes),
)

for name, operation, target in quantization_nodes:
    print(f"{name:45s} {operation:15s} {target}")

In [ ]:
for node in quantized_model.graph.nodes:
    target = str(node.target)

    if any(
        keyword in target
        for keyword in [
            "linear",
            "mm",
            "addmm",
            "matmul",
            "quantize",
            "dequantize",
        ]
    ):
        print(
            f"{node.name:45s} "
            f"{node.op:15s} "
            f"{target}"
        )

In [ ]:
quantized_edge_model = litert_torch.convert(
    quantized_model,
    sample_inputs,
    quant_config=QuantConfig(
        pt2e_quantizer=quantizer
    ),
)

print("Quantized LiteRT conversion succeeded")

In [ ]:
quantized_litert_output = quantized_edge_model(
    *sample_inputs
)

if isinstance(
    quantized_litert_output,
    (tuple, list),
):
    quantized_litert_output = (
        quantized_litert_output[0]
    )

quantized_litert_output = np.asarray(
    quantized_litert_output
)

pt2e_output_numpy = (
    quantized_torch_output
    .detach()
    .cpu()
    .numpy()
)

conversion_error = np.abs(
    pt2e_output_numpy
    - quantized_litert_output
)

print(
    "Quantized LiteRT output shape:",
    quantized_litert_output.shape,
)

print(
    "PT2E versus LiteRT maximum error:",
    conversion_error.max(),
)

print(
    "PT2E versus LiteRT mean error:",
    conversion_error.mean(),
)

In [ ]:
quantized_output_path = (
    output_directory
    / "tinystories_block0_w8a8.tflite"
)

quantized_edge_model.export(
    str(quantized_output_path)
)

print(
    "Saved quantized model:",
    quantized_output_path.resolve(),
)

print(
    "FP32 LiteRT size:",
    output_path.stat().st_size,
    "bytes",
)

print(
    "W8A8 LiteRT size:",
    quantized_output_path.stat().st_size,
    "bytes",
)